# Build an agent with custom model serving and Modal Sandboxes

This tutorial builds a small tool-calling agent from ordinary Python and
Modal primitives. A Qwen3-8B model runs behind a custom SGLang
`@app.server`, while every tool call executes inside an isolated
[Modal Sandbox](https://modal.com/docs/guide/sandbox).

You will:
1. Serve Qwen3-8B with a custom Modal Server.
2. Create a Sandbox and populate its filesystem.
3. Expose directory listing and file reading as tools.
4. Run an OpenAI-compatible tool-calling loop.
5. Terminate the Sandbox even if the agent fails.

No deployment or evaluation framework is involved. The server, tool
dispatcher, agent loop, and cleanup are all visible in this tutorial.

In [ ]:
import importlib.util

# Skip if modal_training_gym is already importable (e.g. a local editable
# checkout) so your edits keep taking effect and the env stays synced.
if importlib.util.find_spec('modal_training_gym') is None:
    %uv pip install -q git+https://github.com/modal-projects/training-gym.git@main

In [ ]:
import json
import posixpath
import subprocess
import time
import urllib.error
import urllib.request

import modal

from modal_training_gym import (
    Qwen3_8B,
    endpoint_chat_message,
    wait_for_server_url,
)

## Serve the model with custom code

Qwen3-8B is not in the managed Endpoint catalog, so we launch SGLang
directly with `@app.server`. The Hugging Face cache is a Modal Volume.
The SGLang image already contains a cache directory, so the image build
removes it before Modal mounts the Volume at the same path.

SGLang's `qwen` parser converts the model's tool-call syntax into the
OpenAI-compatible `tool_calls` field consumed by the loop below.

In [ ]:
MODEL_ID = Qwen3_8B().model_name
SERVER_APP_NAME = "gym-qwen3-8b-agent"
SERVER_PORT = 8000
SERVER_STARTUP_TIMEOUT = 20 * 60

server_image = (
    modal.Image.from_registry("lmsysorg/sglang:v0.5.12")
    .entrypoint([])
    .run_commands("rm -rf /root/.cache/huggingface")
    .env({"HF_HUB_CACHE": "/root/.cache/huggingface"})
)

def serve_model() -> str:
    app = modal.App(SERVER_APP_NAME)

    @app.server(
        image=server_image,
        gpu="H100",
        volumes={
            "/root/.cache/huggingface": modal.Volume.from_name(
                "huggingface-cache", create_if_missing=True
            )
        },
        port=SERVER_PORT,
        startup_timeout=SERVER_STARTUP_TIMEOUT,
        scaledown_window=10 * 60,
        exit_grace_period=25,
        target_concurrency=4,
        unauthenticated=True,
        serialized=True,
    )
    class ModelServer:
        @modal.enter()
        def start(self):
            command = [
                "python",
                "-m",
                "sglang.launch_server",
                "--model-path",
                MODEL_ID,
                "--served-model-name",
                MODEL_ID,
                "--host",
                "0.0.0.0",
                "--port",
                str(SERVER_PORT),
                "--mem-fraction-static",
                "0.82",
                "--context-length",
                "32768",
                "--tool-call-parser",
                "qwen",
                "--trust-remote-code",
            ]
            print(" ".join(command), flush=True)
            self.proc = subprocess.Popen(command)

            deadline = time.monotonic() + SERVER_STARTUP_TIMEOUT
            health = f"http://127.0.0.1:{SERVER_PORT}/health"
            while time.monotonic() < deadline:
                if self.proc.poll() is not None:
                    raise RuntimeError(
                        f"SGLang exited with code {self.proc.returncode}"
                    )
                try:
                    with urllib.request.urlopen(health, timeout=5) as response:
                        if response.status == 200:
                            return
                except (urllib.error.URLError, TimeoutError, OSError):
                    time.sleep(2)
            raise TimeoutError(f"SGLang not healthy at {health}")

        @modal.exit()
        def stop(self):
            process = getattr(self, "proc", None)
            if process is not None and process.poll() is None:
                process.terminate()
                process.wait(timeout=30)

    with modal.enable_output():
        app.deploy()
    return wait_for_server_url(ModelServer, label="Qwen3-8B agent server")

model_url = serve_model()
print(f"Model URL: {model_url}")

## Define the tools

The model receives two function schemas. The dispatcher maps each function
name to an argument vector and passes the requested path as a separate
process argument. Tool paths are restricted to `/repo`, which is the only
project tree the agent should inspect.

In [ ]:
TOOL_DEFINITIONS = [
    {
        "type": "function",
        "function": {
            "name": "list_directory",
            "description": "List one directory under /repo.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Absolute directory path under /repo.",
                    }
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read one text file under /repo.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Absolute file path under /repo.",
                    }
                },
                "required": ["path"],
            },
        },
    },
]
TOOL_COMMANDS = {
    "list_directory": ("ls", "-1F"),
    "read_file": ("cat",),
}

def dispatch_tool(sandbox: modal.Sandbox, tool_call: dict) -> str:
    function = tool_call.get("function") or {}
    name = function.get("name")
    command = TOOL_COMMANDS.get(name)
    if command is None:
        raise ValueError(f"Unknown tool: {name!r}")

    arguments = json.loads(function.get("arguments") or "{}")
    raw_path = arguments.get("path")
    path = posixpath.normpath(raw_path) if isinstance(raw_path, str) else ""
    if not (path == "/repo" or path.startswith("/repo/")):
        raise ValueError(f"Tool path must resolve under /repo: {raw_path!r}")

    process = sandbox.exec(*command, path)
    stdout = process.stdout.read()
    stderr = process.stderr.read()
    process.wait()
    if process.returncode != 0:
        raise RuntimeError(f"{name}: {stderr.strip()}")
    return stdout

## Create the Sandbox

The Sandbox starts with a small Python project. `sleep infinity` keeps its
entrypoint alive while the agent executes tools. The 10-minute timeout is a
final billing guard if the local process disappears before cleanup.

In [ ]:
FILES = {
    "/repo/README.md": (
        "# Fibonacci utility\n\n"
        "Run `python fib.py 10` to print the first ten Fibonacci numbers.\n"
    ),
    "/repo/fib.py": (
        "import sys\n\n\n"
        "def fibonacci(n: int) -> list[int]:\n"
        "    if n <= 0:\n"
        "        return []\n"
        "    sequence = [0, 1]\n"
        "    while len(sequence) < n:\n"
        "        sequence.append(sequence[-1] + sequence[-2])\n"
        "    return sequence[:n]\n\n\n"
        "if __name__ == '__main__':\n"
        "    count = int(sys.argv[1]) if len(sys.argv) > 1 else 10\n"
        "    print(fibonacci(count))\n"
    ),
    "/repo/tests/test_fib.py": (
        "from fib import fibonacci\n\n\n"
        "def test_empty():\n"
        "    assert fibonacci(0) == []\n\n\n"
        "def test_ten():\n"
        "    assert fibonacci(10)[-1] == 34\n"
    ),
}

sandbox_app = modal.App.lookup("agent-sandbox-tutorial", create_if_missing=True)
with modal.enable_output():
    sandbox = modal.Sandbox.create(
        "sleep",
        "infinity",
        app=sandbox_app,
        image=modal.Image.debian_slim(python_version="3.12"),
        timeout=10 * 60,
    )

sandbox.filesystem.make_directory("/repo/tests")
for path, content in FILES.items():
    sandbox.filesystem.write_text(content, path)
print(f"Sandbox created: {sandbox.object_id}")

## Run the agent loop

Every model response is an OpenAI-compatible message dictionary. When it
contains `tool_calls`, the loop executes them and appends matching `tool`
messages. A response without tool calls is the final answer.

Cleanup lives in `finally`, so model, network, parsing, and tool failures all
terminate the Sandbox.

In [ ]:
messages = [
    {
        "role": "user",
        "content": (
            "Explore /repo. List its files, read each file, then summarize "
            "what the project does and whether its tests look correct."
        ),
    }
]

try:
    for iteration in range(10):
        message = endpoint_chat_message(
            model_url,
            model=MODEL_ID,
            messages=messages,
            tools=TOOL_DEFINITIONS,
            max_tokens=2048,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        messages.append(message)
        tool_calls = message.get("tool_calls") or []

        if not tool_calls:
            content = (
                message.get("content") or message.get("reasoning_content") or ""
            )
            print(f"Agent response:\n{content}")
            break

        for tool_call in tool_calls:
            function = tool_call["function"]
            print(
                f"[{iteration + 1}] {function['name']}"
                f"({function.get('arguments', '{}')})"
            )
            result = dispatch_tool(sandbox, tool_call)
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call["id"],
                    "content": result,
                }
            )
    else:
        raise RuntimeError("Agent reached the 10-iteration limit")
finally:
    sandbox.terminate(wait=True)
    print("Sandbox terminated.")

## Next steps

This agent is intentionally small. Extend it with a command runner, file
writes, or a filesystem snapshot. Keep the same boundaries: the model
proposes typed tool calls, the dispatcher validates them, and untrusted
execution stays inside the Sandbox.